# Phase 5 — Pipeline Ablation

Per spec 4.9 — the **system-level** ablation:

| Variant | Components |
|---|---|
| A. churn-only | Highest p(churn) customers, recommend most-popular items |
| B. + retrieval | SASRec top-K, ignore churn for ranking |
| C. + retrieval + ranking | SASRec retrieval reranked by LGBMRanker (using p(churn) as feature) |
| D. + LLM reranker | C, with LLM rerank on top-20 |

We measure Recall@10 / NDCG@10 on the high-churn segment (top-20% p(churn)) — what the system is *meant* to optimize.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import os
import numpy as np
import pandas as pd

from src.data.splits import build_windows
from src.evaluation.recsys_metrics import evaluate_recsys
from src.models.ranking.lgbm_ranker import build_pairs, train_ranker, rerank
from src.features.build_features import build_customer_features

PROC = PROJECT_ROOT / 'data' / 'processed'
FEAT = PROJECT_ROOT / 'data' / 'features'
REPORTS = PROJECT_ROOT / 'reports'
df = pd.read_parquet(PROC / 'transactions_clean.parquet')
labels = {n: pd.read_parquet(PROC / f'churn_labels_{n}.parquet') for n in ['train','val','test']}
candidates = pd.read_parquet(FEAT / 'retrieval_candidates_test.parquet')
churn_gbm = pd.read_parquet(FEAT / 'churn_scores_gbm_test.parquet')
windows = build_windows(df, horizon_days=90, n_windows=3)
w_test = windows[2]

In [ ]:
# Define the high-churn segment & label-window purchases
p80 = churn_gbm['p_churn_gbm'].quantile(0.8)
high_risk = churn_gbm[churn_gbm['p_churn_gbm'] >= p80]['customer_id'].tolist()
print(f'High-risk customers (top 20% p_churn): {len(high_risk):,}')

lw = df[(df['invoice_date'] >= w_test.feature_end) & (df['invoice_date'] < w_test.label_end)]
gt = lw[lw['customer_id'].isin(high_risk)][['customer_id','stock_code']]
print(f'Ground-truth purchases in label window from high-risk customers: {len(gt):,}')

## Variant A — Churn-only (recommend globally most popular items)

In [ ]:
pre = df[df['invoice_date'] < w_test.feature_end]
popular = pre['stock_code'].value_counts().head(10).index.tolist()
rec_A = pd.DataFrame([(c, sc) for c in high_risk for sc in popular], columns=['customer_id','stock_code'])
m_A = evaluate_recsys(rec_A, gt, ks=(5, 10))
m_A

## Variant B — SASRec retrieval only (top-10)

In [ ]:
rec_B = (candidates[candidates['customer_id'].isin(high_risk)]
         .sort_values(['customer_id','rank']).groupby('customer_id').head(10)
         [['customer_id','stock_code']])
m_B = evaluate_recsys(rec_B, gt, ks=(5, 10))
m_B

## Variant C — Retrieval + LGBM ranker (using p(churn) as feature)

In [ ]:
top_countries = df['country'].value_counts().head(8).index.tolist()
cust_feats = build_customer_features(df, w_test.feature_end, country_dummies=top_countries)
item_feats = (pre.groupby('stock_code').agg(
    item_n_orders=('invoice','nunique'),
    item_n_customers=('customer_id','nunique'),
    item_avg_price=('price','mean'),
).reset_index())
item_feats['item_log_orders'] = np.log1p(item_feats['item_n_orders'])
churn_for_join = churn_gbm.rename(columns={'p_churn_gbm':'p_churn'})

pairs = build_pairs(candidates, lw, cust_feats, item_feats, churn_for_join)
feature_cols = [c for c in ['score','log_score','inv_rank','rank','p_churn',
                            'recency_days','frequency','monetary','mean_interval_days',
                            'product_diversity','item_log_orders','item_n_customers','item_avg_price']
                if c in pairs.columns]

# split by customer
cust_ids = pairs['customer_id'].drop_duplicates().sample(frac=1, random_state=42).tolist()
n_tr = int(0.8 * len(cust_ids))
tr_cust = set(cust_ids[:n_tr])
ranker = train_ranker(pairs[pairs['customer_id'].isin(tr_cust)], feature_cols)

# rerank for high-risk customers only
hr_pairs = pairs[pairs['customer_id'].isin(high_risk)]
rec_C = rerank(ranker, hr_pairs, feature_cols, k=10)[['customer_id','stock_code']]
m_C = evaluate_recsys(rec_C, gt, ks=(5, 10))
m_C

## Variant D — Add LLM reranker on top-20 (optional, requires ANTHROPIC_API_KEY)

In [ ]:
m_D = None
if os.environ.get('ANTHROPIC_API_KEY'):
    from src.models.reranker.llm import LLMReranker, LLMRerankerConfig
    # For speed in this ablation we only LLM-rerank for a small sample
    sample_users = list(set(high_risk))[:30]
    rer = LLMReranker(LLMRerankerConfig(top_k_to_rerank=20))
    rec_D_rows = []
    for cust in sample_users:
        hist = df[df['customer_id']==cust].tail(30)[['stock_code','description']]
        cand = (rec_C[rec_C['customer_id']==cust].head(20).merge(
            pre[['stock_code','description','price']].drop_duplicates('stock_code'),
            on='stock_code', how='left'))
        if cand.empty: continue
        cand['score'] = 0.0  # not used
        try:
            ordered = rer.rerank(hist, cand)
            for sc in ordered[:10]:
                rec_D_rows.append((cust, sc))
        except Exception as e:
            print(f'LLM rerank failed for {cust}: {e}')
    rec_D = pd.DataFrame(rec_D_rows, columns=['customer_id','stock_code'])
    gt_D = gt[gt['customer_id'].isin(sample_users)]
    m_D = evaluate_recsys(rec_D, gt_D, ks=(5, 10)) if len(rec_D) else {'note': 'no_recs'}
else:
    print('No ANTHROPIC_API_KEY — skipping Variant D')
m_D

## Final ablation table

In [ ]:
rows = {
    'A. churn-only (popular)': m_A,
    'B. + SASRec retrieval': m_B,
    'C. + LGBM ranker (uses p_churn)': m_C,
}
if m_D is not None and 'recall@5' in m_D:
    rows['D. + LLM reranker (30-user sample)'] = m_D
table = pd.DataFrame(rows).T.round(4)
table.to_csv(REPORTS / 'pipeline_ablation.csv')
table